# Import Library

In [4]:
import pandas as pd
import re
import string
import nltk
import joblib

from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.base import BaseEstimator, TransformerMixin
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load Data

In [5]:
df = pd.read_excel("jamu206.xlsx")

df

,NO,NAMA JAMU,KHASIAT,KANDUNGAN,ATURAN MINUM,EFEK SAMPING,JENIS,PRODUSEN,LOKASI PEMASARAN,LOKASI PRODUKSI,KABUPATEN,PERIZINAN
0,1,Galian Rapat Wangi,"mengurangi bau badan, mengurangi bau tidak sed...","kunci, kunyit, pinang muda parabes",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
1,2,Empot-empot legit,"Mengurangi lendir berlebih, mengatasi keputiha...","kunci, kunyit, pinang muda parabes",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
2,3,galian singset,"mengurangi lemak dalam tubuh, menambah nafsu m...","kunyit, daun sirih, delima putih",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
3,4,jamu kecantikan,"perawatan khusus remaja putri, mengurangi bau ...","daun sirih, kunyit, kulit manggis, kunci, pina...",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
4,5,jamu galian montok,untuk menambah nafsu makan,"temulawak , jahe",NaN,tidak ada,pil,Madura Sari,Jl Pahlawan 21 Sampang,Jl Pahlawan 21 Sampang,Sampang,KEMENKES
...,...,...,...,...,...,...,...,...,...,...,...,...
201,202,galian rapet,rapat mencengkram saat berhubungan intim.hingg...,NaN,3 x sehari @5 pil sekali minum,NaN,pil,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM
202,203,jamu dalimah putih,dan selalu awet muda,NaN,3 x sehari @ 5 biji,NaN,pil,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM
203,204,GALIAN SEHAT / MONTOK,Pil jamu ramuan Madura untuk wanita / pria yan...,"Rimpang Kunyit,Rimpang Temulawak, Rimpang Leng...",Minum 3 x 5 pil / hari.,NaN,pil,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM
204,205,minyak zaitun,Mangencangkan dan menghaluskan kulit muka dan ...,NaN,Oleskan minyak zaitun secukupnya bagian kulit/...,NaN,cair,berkah madu/sumber madu,indonesia,"Jl. KH. Mohamad Kholil No.50A, Demangan Timur,...",bangkalan,BPOM


In [6]:
print(df.isnull().sum())
df = df.dropna(subset=['KHASIAT'])
print("Jumlah data setelah drop:", len(df))

NO                   0
NAMA JAMU            0
KHASIAT              4
KANDUNGAN           82
ATURAN MINUM        93
EFEK SAMPING        48
JENIS               28
PRODUSEN             0
LOKASI PEMASARAN     8
LOKASI PRODUKSI     23
KABUPATEN            3
PERIZINAN           87
dtype: int64
Jumlah data setelah drop: 202


# Labeling

In [7]:
def labeling(text):
    text = str(text).lower()

    if any(k in text for k in [
        'stamina','tenaga','energi','lelah','capek','letih','lesu','loyo',
        'ejakulasi','perkasa','kejantanan','sperma','vitalitas','daya tahan',
        'kuat','kebugaran','fit','tidak mudah capek','menambah tenaga',
        'bab','buang air besar','nafsu makan','maag','asam lambung',
        'lemak','diet','kolesterol','berat badan','melangsingkan',
        'pelangsing','gemuk','kurus','metabolisme','pencernaan',
        'kembung','mual','perut','lambung','pegal','linu',
        'nyeri','rematik','encok','sendi','otot',
        'demam','flu','batuk','pilek',
        'diabetes','hipertensi','imun',
        'asma','ginjal','jantung','radang'
    ]):
        return 'stamina'

    elif any(k in text for k in [
        'keputihan','vagina','rahim','haid','menstruasi',
        'asi','kandungan','organ intim','hormonal'
    ]):
        return 'kesehatan_wanita'

    elif any(k in text for k in [
        'kulit','wajah','jerawat','komedo','cerah','halus',
        'glowing','keriput','flek','cantik',
        'gatal','alergi','jamur','eksim','panu',
        'bau badan','bau mulut','keringat'
    ]):
        return 'perawatan'

    else:
        return 'lainnya'

df['label'] = df['KHASIAT'].apply(labeling)

/tmp/ipykernel_4571/2062752412.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['KHASIAT'].apply(labeling)


# Preprocessing

In [8]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stop_words = set(stopwords.words('indonesian'))

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.clean_text(text) for text in X]

    def clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\d+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text).strip()
        tokens = text.split()
        tokens = [w for w in tokens if w not in self.stop_words]
        return " ".join(tokens)

# Split Data

In [9]:
X = df['KHASIAT']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.1,
    random_state=42,
    stratify=y
)

# Modelling

In [10]:
pipeline = Pipeline([
    ('prep', TextPreprocessor()),
    ('tfidf', TfidfVectorizer()),
    ('ros', RandomOverSampler(random_state=42)),
    ('nb', MultinomialNB())
])

In [11]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('prep', TextPreprocessor()), ('tfidf', TfidfVectorizer()),
                ('ros', RandomOverSampler(random_state=42)),
                ('nb', MultinomialNB())])

# Evaluasi Model

In [12]:
y_pred = pipeline.predict(X_test)
print("\n=== HASIL EVALUASI ===")
print(classification_report(y_test, y_pred))


=== HASIL EVALUASI ===
                  precision    recall  f1-score   support

kesehatan_wanita       0.80      1.00      0.89         4
         lainnya       1.00      0.25      0.40         4
       perawatan       0.50      0.50      0.50         2
         stamina       0.77      0.91      0.83        11

        accuracy                           0.76        21
       macro avg       0.77      0.66      0.66        21
    weighted avg       0.79      0.76      0.73        21



# Simpan Model

In [13]:
joblib.dump(pipeline, "model_pipeline.pkl")

print("\nModel berhasil disimpan sebagai model_pipeline.pkl")


Model berhasil disimpan sebagai model_pipeline.pkl


# **Prediksi**

In [14]:
import joblib

model = joblib.load("model_pipeline.pkl")

text = "perut kembung dan nyeri badan"

label = model.predict([text])[0]
prob  = model.predict_proba([text]).max()

print("Input      :", text)
print("Prediksi   :", label)
print("Confidence :", prob)

Input      : perut kembung dan nyeri badan
Prediksi   : stamina
Confidence : 0.5458764098897988
